This notebook takes the NAICS Employed, Quarterly Vac, and Monthly Vac of Canada from the S3 folder, checks for nulls, dupes, schema and clean/finalizes them, then transforms Monthly and Quarterly Vac into pivot tables with an addition of "Employed" from NAICS Employed and computes vacancies over employed * 1000 (vacancies_per_1000) as a normalized vacancy rate. Finally, exporting the pivot tables and cleaned NAICS Employed back into S3

In [0]:
from pyspark.sql.functions import first, col, split, trim, regexp_extract, sum as spark_sum, when, countDistinct, to_date, month as month_fn

from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("base_input_path", "s3a://your-bucket/processed/")  # please change to your own bucket
dbutils.widgets.text("base_output_path", "s3a://your-bucket/curated/")  # please change to your own bucket

base_input = dbutils.widgets.get("base_input_path").rstrip("/")
base_output = dbutils.widgets.get("base_output_path").rstrip("/")

df_Monthly  = spark.read.parquet(f"{base_input}/monthly_vacancies_processed//")
df_NAICS    = spark.read.parquet(f"{base_input}/naics_employed_processed//")
df_Quarterly = spark.read.parquet(f"{base_input}/quarterly_vacancies_processed//")

<h1>Cleaning Data</h1>

<h2>Monthly</h2>

In [0]:
df_Monthly.printSchema()
display(df_Monthly.limit(20))
display(df_Monthly.select("Date").distinct().orderBy("Date").limit(200))
display(df_Monthly.select("Statistics").distinct().orderBy("Statistics").limit(5))

In [0]:
print("total rows:", df_Monthly.count())

# null counts per column
nulls_Monthly = df_Monthly.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_Monthly.columns])
display(nulls_Monthly)

print("distinct GEO:", df_Monthly.select("GEO").distinct().count())
print("distinct Year:", df_Monthly.select("Year").distinct().count())
print("distinct Statistics:", df_Monthly.select("Statistics").distinct().count())

In [0]:
# duplicates by GEO+Year+Statistics
dupes1_Monthly = df_Monthly.groupBy("GEO","Year","Statistics") \
                  .count().filter("count > 1").orderBy("count", ascending=True) # Lowest count to check for amount of missing values
display(dupes1_Monthly.limit(200))
print("dup groups (GEO,Year,Statistics):", dupes1_Monthly.count())

# duplicates by GEO+Date+Statistics
dupes2_Monthly = df_Monthly.groupBy("GEO","Date","Statistics").count().filter("count > 1").orderBy("count", ascending=True)
display(dupes2_Monthly.limit(20))
print("dup groups (GEO,Date,Statistics):", dupes2_Monthly.count())


In [0]:
clean_Monthly = df_Monthly.withColumn("Date_parsed", to_date(col("Date"), "yyyy-MM"))
display(clean_Monthly.select("Date", "Date_parsed").distinct().orderBy("Date").limit(50))
display(clean_Monthly.select("Date","Date_parsed").limit(50))

<h2>Quarterly</h2>

In [0]:
df_Quarterly.printSchema()
display(df_Quarterly.limit(20))
display(df_Quarterly.select("Date").distinct().orderBy("Date").limit(200))
display(df_Quarterly.select("Statistics").distinct().orderBy("Statistics").limit(5))
display(df_Quarterly.select("NAICS").distinct().orderBy("NAICS").limit(10))

In [0]:
print("total rows:", df_Quarterly.count())

# null counts per column
nulls_Quarterly = df_Quarterly.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_Quarterly.columns])
display(nulls_Quarterly)

print("distinct GEO:", df_Quarterly.select("GEO").distinct().count())
print("distinct Year:", df_Quarterly.select("Year").distinct().count())
print("distinct Statistics:", df_Quarterly.select("Statistics").distinct().count())
print("distinct NAICS:", df_Quarterly.select("NAICS").distinct().count())

In [0]:
# duplicates by GEO+Year+Statistics+NAICS
dupes1_Quarterly = df_Quarterly.groupBy("GEO","Year","Statistics","NAICS") \
                  .count().filter("count > 1").orderBy("count", ascending=True) # Lowest count to check for amount of missing values
display(dupes1_Quarterly.limit(500))
print("dup groups (GEO,Year,Statistics):", dupes1_Quarterly.count())

# duplicates by GEO+Date+Statistics+NAICS
dupes2_Quarterly = df_Quarterly.groupBy("GEO","Date","Statistics","NAICS").count().filter("count > 1").orderBy("count", ascending=True)
display(dupes2_Quarterly.limit(20))
print("dup groups (GEO,Date,Statistics):", dupes2_Quarterly.count())


In [0]:
clean_Quarterly = df_Quarterly.withColumn("Date_parsed", to_date(col("Date"), "yyyy-MM"))
display(clean_Quarterly.select("Date", "Date_parsed").distinct().orderBy("Date").limit(50))
display(clean_Quarterly.select("Date","Date_parsed").limit(50))

<h2>NAICS</h2>

In [0]:
df_NAICS.printSchema()
display(df_NAICS.limit(20))
display(df_NAICS.select("Date").distinct().orderBy("Date").limit(200))
display(df_NAICS.select("NAICS").distinct().orderBy("NAICS").limit(5))

In [0]:
print("total rows:", df_NAICS.count())

# null counts per column
nulls_NAICS = df_NAICS.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_NAICS.columns])
display(nulls_NAICS)

print("distinct GEO:", df_NAICS.select("GEO").distinct().count())
print("distinct Year:", df_NAICS.select("Year").distinct().count())
print("distinct NAICS:", df_NAICS.select("NAICS").distinct().count())

In [0]:
# duplicates by GEO+Year+Statistics
dupes1_NAICS = df_NAICS.groupBy("GEO","Year","NAICS") \
                  .count().filter("count > 1").orderBy("count", ascending=True) # Lowest count to check for amount of missing values
print("dup groups (GEO,Year,Statistics):", dupes1_NAICS.count())

# duplicates by GEO+Date+Statistics
dupes2_NAICS = df_NAICS.groupBy("GEO","Date","NAICS").count().filter("count > 1").orderBy("count", ascending=True)
display(dupes2_NAICS.limit(20))
print("dup groups (GEO,Date,Statistics):", dupes2_NAICS.count())


In [0]:
clean_NAICS = df_NAICS.withColumn("Date_parsed", to_date(col("Date"), "yyyy-MM"))
clean_NAICS = clean_NAICS.withColumn("month_parsed", month_fn(col("date_parsed")))
display(clean_NAICS.select("Date", "Date_parsed").distinct().orderBy("Date").limit(20))
display(clean_NAICS.select("Date","Date_parsed","month_parsed").limit(20))
display(clean_NAICS.head(10))

<h1>Computing metrics</h1>

<h2>Monthly Vac + Employed + Vac over 1000</h2>

In [0]:
# Taking out the total number of employed people in Canada and renaming to standardize the name
Filtered_NAICS = clean_NAICS.filter(
    col("NAICS") == "Industrial aggregate including unclassified businesses [00-91N]").withColumnRenamed("VALUE", "Total Employment")

pivoted_Monthly_Vac = clean_Monthly.groupBy("Year","Date_parsed", "GEO") \
    .pivot("Statistics") \
    .sum("VALUE")

# Adding the total number of employed people in Canada to the pivoted table
pivoted_Monthly_Vac = pivoted_Monthly_Vac.join(
    Filtered_NAICS.select("Date_parsed", "Total Employment","GEO"),
    on=["Date_parsed", "GEO"],
    how="left"
)

# Standardizing vacancies against employment
pivoted_Monthly_Vac = pivoted_Monthly_Vac.withColumn(
    "Vacancies_per_1000",
    when(col("Total Employment") != 0,
         (col("Job vacancies") / col("Total Employment")) * 1000 
    ).otherwise(None) 
)

In [0]:
# Checking for dupes
clean_Monthly.groupBy(
    "Date_parsed", "GEO", "Statistics"
).count().filter("count > 1").show()

In [0]:
pivoted_Monthly_Vac.orderBy("Date_parsed").show(10)

In [0]:
# Checking for nulls in each column
display(pivoted_Monthly_Vac.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in pivoted_Monthly_Vac.columns]))

<h2>Quarter Vac + Employed + Vac over 1000</h2>

In [0]:
NAICS_with_Quarter_label = (clean_NAICS
                    .withColumn("quarter_label",
                    F.date_format(F.date_trunc("quarter", F.col("Date_parsed")), "yyyy-MM"))
)

NAICS_in_Quarters = (
    NAICS_with_Quarter_label.groupBy("quarter_label", "NAICS", "GEO", "Year")
                            .agg(
                                F.max(
                                    F.when(
                                        F.col("VALUE").isNotNull(),
                                        F.struct(F.col("Date_parsed").cast("timestamp"), F.col("VALUE"))
                                    )
                                ).alias("latest_non_null"), # Struct (Date_parsed, VALUE) for when value is not null taking the latest date & Value
                                F.min("Date_parsed").alias("Date_parsed")
                             )
                            .withColumn("VALUE", F.col("latest_non_null").getField("VALUE")) # extract the latest VALUE from the struct
                            .drop("latest_non_null")
                            .withColumnRenamed("quarter_label", "Date")
                            .orderBy("Date")
)
                        

NAICS_in_Quarters.show(truncate=False)

In [0]:
display(NAICS_with_Quarter_label.filter(col("Date_parsed") >= "2022-01-01").orderBy("Date_parsed"))

In [0]:
clean_NAICS2 = NAICS_in_Quarters.withColumn(
    "NAICS",
    when(
        col("NAICS") == "Industrial aggregate including unclassified businesses [00-91N]",
        "Total, all industries" # Standardizing the name to match Quarterly Vac
    ).otherwise(col("NAICS")))

# Takes the NAICS names without the codes eg [11]
clean_NAICS2 = clean_NAICS2.withColumn(
    "NAICS_name",
    trim(split(col("NAICS"), r"\s*\[").getItem(0))
)

clean_Quarterly2 = clean_Quarterly.withColumn(
    "NAICS_name",
    trim(split(col("NAICS"), r"\s*\[").getItem(0))
)

In [0]:
# Check which NAICS are either missing or mismatched (The Agriculture NAICS has no corresponding employed and is intended)
missing_in_NAICS = clean_Quarterly2.join(clean_NAICS2, on=["NAICS_name","Date", "GEO"], how="left_anti")
missing_in_NAICS.show(50, truncate=False)

print(f"Names in Quarterly but missing in NAICS:{missing_in_NAICS.count()} out of {clean_Quarterly2.count()}")

print("distinct missing NAICS:", missing_in_NAICS.select("NAICS_name").distinct().count())

In [0]:
# Taking only needed columns
clean_NAICS_ready = (
    clean_NAICS2
        .withColumn("Statistics", F.lit("Total Employment"))
        .select("Year","Date", "Date_parsed","NAICS","NAICS_name", "GEO", "Statistics", "VALUE")
)

clean_NAICS_filtered = clean_NAICS_ready.join(
    clean_Quarterly2.select("NAICS_name").distinct(),
    on="NAICS_name",
    how="left_semi"
)

clean_Quarterly_ready = clean_Quarterly2.select(
    "Year","Date", "NAICS", "Date_parsed","NAICS_name", "GEO", "Statistics", "VALUE"
)

Quarterly_employed = clean_Quarterly_ready.unionByName(clean_NAICS_filtered, allowMissingColumns=True) # allow miissing columns to see data gaps later 

display(Quarterly_employed)

In [0]:
display(Quarterly_employed.filter(
    F.col("Statistics") == "Total Employment"
))

In [0]:
# Check for dupes
Quarterly_employed.groupBy(
    "NAICS","Date_parsed", "GEO", "Statistics"
).count().filter("count > 1").show()

In [0]:
pivoted_Quarterly_Vac = Quarterly_employed.groupBy("Year","Date_parsed", "GEO", "NAICS") \
    .pivot("Statistics") \
    .sum("VALUE")

# Standardizing vacancies against employment
pivoted_Quarterly_Vac_with_metric = pivoted_Quarterly_Vac.withColumn(
    "Vacancies_per_1000",
    when(col("Total Employment") != 0,
         (col("Job vacancies") / col("Total Employment")) * 1000
    ).otherwise(None)
)

In [0]:
Quarterly_employed.select("Statistics").distinct().orderBy("Statistics").show(truncate=False)

Quarterly_employed.groupBy("Date_parsed","GEO","NAICS","Statistics") \
    .count().filter("count > 1").show(50, truncate=False)

In [0]:
display(clean_NAICS_filtered.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in clean_NAICS_filtered.columns]))

In [0]:
display(clean_Quarterly_ready.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in clean_Quarterly_ready.columns]))

In [0]:
display(pivoted_Quarterly_Vac_with_metric.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in pivoted_Quarterly_Vac_with_metric.columns]))

In [0]:
display(pivoted_Quarterly_Vac_with_metric.orderBy("NAICS", "Date_parsed", "GEO"))

In [0]:
display(pivoted_Quarterly_Vac_with_metric 
    .select(
        "Date_parsed",
        "GEO",
        "NAICS",
        "Average offered hourly wage",
        "Job vacancy rate",
        "Job vacancies",
        "Total Employment"
    ) 
    .filter(
        (F.col("Total Employment").isNull()) &
        (
            F.col("Average offered hourly wage").isNotNull() |
            F.col("Job vacancy rate").isNotNull() |
            F.col("Job vacancies").isNotNull()
        )
    )
    .groupBy("Date_parsed").count().orderBy("count")
)


<h1>Exporting to S3</h1>

In [0]:
# Cleaned NAICS with the Schema
clean_NAICS.write.mode("overwrite").partitionBy("Year", "GEO").parquet(f"{base_output}/naics_employed_curated/")

# Monthly Vac and Employed
pivoted_Monthly_Vac.write.mode("overwrite").partitionBy("Year", "GEO").parquet(f"{base_output}/monthly_vacancies_employed_curated/")

# Quarterly Vac and Employed
pivoted_Quarterly_Vac_with_metric.write.mode("overwrite").partitionBy("Year", "GEO").parquet(f"{base_output}/quarterly_vacancies_employed_curated/")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7596193229016327>, line 8
      5 pivoted_Monthly_Vac.write.mode("overwrite").partitionBy("Year", "GEO").parquet(f"{base_output}/monthly_vacancies_employed_curated/")
      7 # Quarterly Vac and Employed
----> 8 pivoted_Quarterly_Vac_with_metric.write.mode("overwrite").partitionBy("Year", "GEO").parquet(f"{base_output}/quarterly_vacancies_employed_curated/")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:779, in DataFrameWriter.parquet(self, path, mode, partitionBy, compression)
    777     self.partitionBy(partitionBy)
    778 self._set_opts(compression=compression)
--> 779 self.format("parquet").save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701   